In [1]:
# Upload datasets as .csv files in your colab session, and then run this cell.
# Dataset Link: https://drive.google.com/drive/folders/1PzURxyRaN5rclRcyRSI9G8oa1XVSo_SF?usp=drive_link

# Don't work in this colab. Make a copy in your drive instead.

import pandas as pd

copd_df = '/content/COPD_data_preprocessed.csv'
alzheimers_df = '/content/alzheimers_disease_data_preprocessed.csv'
arrhythmia_df = '/content/arrhythmia_data_preprocessed.csv'
autism_df = '/content/autism_screening_data_preprocessed.csv'
diabetes_df = '/content/diabetes_data_preprocessed.csv'
stroke_df = '/content/healthcare-dataset-stroke-data_preprocessed.csv'
heart_df = '/content/heart_disease_uci_preprocessed.csv'
hepatitis_df = '/content/hepatitis_data_preprocessed.csv'
liver_df = '/content/liver_patient_dataset_preprocessed.csv'
thyroid_df = '/content/thyroid_dataset_preprocessed.csv'

In [2]:
# Cell 2: Install TabICL and necessary evaluation libraries
!pip install --quiet tabicl scikit-learn statsmodels scipy tqdm pandas numpy

# Modeling goes here..

In [3]:
# Cell: Reusable Single-Dataset Evaluation Function for TabICL
import gc
import os
import warnings
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import statsmodels.api as sm
from tabicl import TabICLClassifier
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

TARGET_COLUMNS = [
    "Dataset",
    "Train_N",
    "Test_N",
    "Num_Feats",
    "Domain",
    "DI_mean",
    "DI_sd",
    "EOD_mean",
    "EOD_sd",
    "DPD_mean",
    "DPD_sd",
    "EqOD_mean",
    "EqOD_sd",
    "PPD_mean",
    "PPD_sd",
    "AUROC_mean",
    "AUROC_sd",
    "Sensitivity_mean",
    "Sensitivity_sd",
    "Specificity_mean",
    "Specificity_sd",
    "PPV_mean",
    "PPV_sd",
    "NPV_mean",
    "NPV_sd",
    "ECE_mean",
    "ECE_sd",
    "NLL_mean",
    "NLL_sd",
    "Brier_mean",
    "Brier_sd",
    "Cal_Slope_mean",
    "Cal_Slope_sd",
    "Cal_Intercept_mean",
    "Cal_Intercept_sd",
    "AUROC_Deg_mean",
    "AUROC_Deg_sd",
    "Accuracy_Deg_mean",
    "Accuracy_Deg_sd",
    "F1_Deg_mean",
    "F1_Deg_sd",
    "Flip_Rate_mean",
    "Flip_Rate_sd",
    "MAPC_mean",
    "MAPC_sd",
]


def run_single_dataset(name, path, domain, output_csv="TabICL_Results.csv"):
    if not os.path.exists(path):
        print(f"Error: {path} not found.")
        return

    print(f"\n---> Running TabICL on: {name} ({domain}) <---")
    df = pd.read_csv(path)

    target_col = df.columns[-1]
    sex_col_name = df.columns[0]
    feature_cols = df.columns[:-1].tolist()
    num_feats = len(feature_cols)

    X = df[feature_cols].copy()
    y = df[target_col].copy().to_numpy().astype(int)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_metrics = []
    train_n_list = []
    test_n_list = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        tqdm(skf.split(X, y), total=5, desc=f"{name} Folds")
    ):
        train_n_list.append(len(train_idx))
        test_n_list.append(len(test_idx))

        X_train_df, y_train = X.iloc[train_idx].copy(), y[train_idx]
        X_test_df, y_test = X.iloc[test_idx].copy(), y[test_idx]

        imputer = KNNImputer(n_neighbors=5)
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(X_train_df), columns=feature_cols
        )
        X_test_imp = pd.DataFrame(imputer.transform(X_test_df), columns=feature_cols)
        sex_test = X_test_imp[sex_col_name].to_numpy().astype(int)

        # TabICL Model
        model = TabICLClassifier(device="cuda", random_state=42)
        model.fit(X_train_imp.values, y_train)

        # Clean
        p_clean = model.predict_proba(X_test_imp.values)[:, 1]
        y_clean_pred = (p_clean >= 0.5).astype(int)

        # Robustness Noise
        X_test_noisy = X_test_imp.copy()
        np.random.seed(42 + fold_idx)
        for col in feature_cols[1:]:
            col_std = np.std(X_train_imp[col].values, ddof=1)
            scale = (
                0.10 * col_std
                if (not np.isnan(col_std) and col_std > 0)
                else 0.10
            )
            X_test_noisy[col] += np.random.normal(
                0, scale, size=len(X_test_noisy)
            )

        # Noisy
        p_noisy = model.predict_proba(X_test_noisy.values)[:, 1]
        y_noisy_pred = (p_noisy >= 0.5).astype(int)

        # Compute Metrics
        # Fairness
        rates = {}
        for grp, grp_name in [(0, "F"), (1, "M")]:
            mask = sex_test == grp
            yt, yp = y_test[mask], y_clean_pred[mask]
            tp = np.sum((yt == 1) & (yp == 1))
            fn = np.sum((yt == 1) & (yp == 0))
            fp = np.sum((yt == 0) & (yp == 1))
            tn = np.sum((yt == 0) & (yp == 0))
            rates[grp_name] = {
                "SR": np.mean(yp == 1) if len(yp) > 0 else np.nan,
                "TPR": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
                "FPR": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
                "PPV": tp / (tp + fp) if (tp + fp) > 0 else np.nan,
            }

        s_m, s_f = rates["M"]["SR"], rates["F"]["SR"]
        t_m, t_f = rates["M"]["TPR"], rates["F"]["TPR"]
        f_m, f_f = rates["M"]["FPR"], rates["F"]["FPR"]
        p_m, p_f = rates["M"]["PPV"], rates["F"]["PPV"]

        di = (
            np.nan
            if (np.isnan(s_m) or np.isnan(s_f) or max(s_m, s_f) == 0)
            else min(s_m, s_f) / max(s_m, s_f)
        )
        eod = np.nan if (np.isnan(t_m) or np.isnan(t_f)) else abs(t_m - t_f)
        dpd = np.nan if (np.isnan(s_m) or np.isnan(s_f)) else abs(s_m - s_f)
        eqod = (
            np.nan
            if (np.isnan(t_m) or np.isnan(t_f) or np.isnan(f_m) or np.isnan(f_f))
            else max(abs(t_m - t_f), abs(f_m - f_f))
        )
        ppd = np.nan if (np.isnan(p_m) or np.isnan(p_f)) else abs(p_m - p_f)

        # Discrimination
        tp = np.sum((y_test == 1) & (y_clean_pred == 1))
        tn = np.sum((y_test == 0) & (y_clean_pred == 0))
        fp = np.sum((y_test == 0) & (y_clean_pred == 1))
        fn = np.sum((y_test == 1) & (y_clean_pred == 0))
        try:
            auroc = roc_auc_score(y_test, p_clean)
        except:
            auroc = np.nan

        # Calibration
        p_c = np.clip(p_clean, 1e-15, 1.0 - 1e-15)
        bins = np.linspace(0.0, 1.0, 16)
        ece = sum(
            (np.sum((p_c >= bins[b]) & (p_c <= bins[b + 1])) / len(y_test))
            * abs(
                np.mean(y_test[(p_c >= bins[b]) & (p_c <= bins[b + 1])])
                - np.mean(p_c[(p_c >= bins[b]) & (p_c <= bins[b + 1])])
            )
            for b in range(15)
            if np.sum((p_c >= bins[b]) & (p_c <= bins[b + 1])) > 0
        )
        nll = -np.mean(y_test * np.log(p_c) + (1 - y_test) * np.log(1 - p_c))
        brier = np.mean((p_c - y_test) ** 2)
        try:
            m_cal = sm.Logit(y_test, sm.add_constant(logit(p_c))).fit(
                disp=False, maxiter=100
            )
            c_int, c_slope = m_cal.params[0], m_cal.params[1]
        except:
            c_int, c_slope = np.nan, np.nan

        # Robustness
        try:
            auroc_deg = auroc - roc_auc_score(y_test, p_noisy)
        except:
            auroc_deg = np.nan
        acc_deg = accuracy_score(y_test, y_clean_pred) - accuracy_score(
            y_test, y_noisy_pred
        )
        f1_deg = f1_score(
            y_test, y_clean_pred, zero_division=0
        ) - f1_score(y_test, y_noisy_pred, zero_division=0)
        flip = np.mean(y_clean_pred != y_noisy_pred)
        mapc = np.mean(np.abs(p_clean - p_noisy))

        fold_metrics.append({
            "DI": di,
            "EOD": eod,
            "DPD": dpd,
            "EqOD": eqod,
            "PPD": ppd,
            "AUROC": auroc,
            "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
            "Specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
            "PPV": tp / (tp + fp) if (tp + fp) > 0 else np.nan,
            "NPV": tn / (tn + fn) if (tn + fn) > 0 else np.nan,
            "ECE": ece,
            "NLL": nll,
            "Brier": brier,
            "Cal_Slope": c_slope,
            "Cal_Intercept": c_int,
            "AUROC_Deg": auroc_deg,
            "Accuracy_Deg": acc_deg,
            "F1_Deg": f1_deg,
            "Flip_Rate": flip,
            "MAPC": mapc,
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    f_df = pd.DataFrame(fold_metrics)
    row = {
        "Dataset": name,
        "Train_N": float(np.mean(train_n_list)),
        "Test_N": float(np.mean(test_n_list)),
        "Num_Feats": int(num_feats),
        "Domain": domain,
    }
    for k in f_df.columns:
        row[f"{k}_mean"] = f_df[k].mean()
        row[f"{k}_sd"] = f_df[k].std(ddof=1)

    # Safely append or update in CSV
    new_row_df = pd.DataFrame([row]).reindex(columns=TARGET_COLUMNS)
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        existing_df = existing_df[existing_df["Dataset"] != name]  # remove old
        combined_df = pd.concat([existing_df, new_row_df], ignore_index=True)
    else:
        combined_df = new_row_df

    combined_df = combined_df.reindex(columns=TARGET_COLUMNS)
    combined_df.to_csv(output_csv, index=False, float_format="%.6f")
    print(f"Done! {name} recorded in {output_csv}.")
    display(new_row_df)

In [4]:
run_single_dataset(
    name="Alzheimers",
    path="/content/alzheimers_disease_data_preprocessed.csv",
    domain="Neurology",
)


---> Running TabICL on: Alzheimers (Neurology) <---


Alzheimers Folds:   0%|          | 0/5 [00:00<?, ?it/s]

INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



tabicl-classifier-v2-20260212.ckpt: reconstructing file:   0%|          |  0.00B /  110MB            

tabicl-classifier-v2-20260212.ckpt: downloading bytes:           |  0.00B            

Done! Alzheimers recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Alzheimers,560.0,140.0,32,Neurology,0.79831,0.137251,0.049427,0.019887,0.11379,...,0.014286,0.013,0.211429,0.030136,0.159776,0.016456,0.248571,0.04855,0.306055,0.028184


In [5]:
run_single_dataset(
    name="Arrhythmia",
    path="/content/arrhythmia_data_preprocessed.csv",
    domain="Cardiology",
)


---> Running TabICL on: Arrhythmia (Cardiology) <---


Arrhythmia Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Arrhythmia recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Arrhythmia,361.6,90.4,262,Cardiology,0.601032,0.068842,0.094648,0.055349,0.207275,...,0.031297,0.013716,0.358486,0.025608,0.159719,0.022868,0.593114,0.059416,0.383486,0.032925


In [6]:
run_single_dataset(
    name="Autism",
    path="/content/autism_screening_data_preprocessed.csv",
    domain="Psychiatry",
)


---> Running TabICL on: Autism (Psychiatry) <---


Autism Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Autism recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Autism,559.2,139.8,19,Psychiatry,0.711783,0.139244,0.0,0.0,0.094228,...,0.0,0.0,0.002857,0.003912,0.005195,0.007113,0.002857,0.003912,0.110344,0.00818


In [7]:

run_single_dataset("COPD", "/content/COPD_data_preprocessed.csv", "Pulmonology")


---> Running TabICL on: COPD (Pulmonology) <---


COPD Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! COPD recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,COPD,504.0,126.0,14,Pulmonology,0.625621,0.106958,0.098738,0.105693,0.316482,...,0.030084,0.019454,0.019048,0.028395,0.013165,0.021195,0.066667,0.024845,0.135854,0.014985


In [8]:
run_single_dataset(
    name="Diabetes",
    path="/content/diabetes_data_preprocessed.csv",
    domain="Endocrinology",
)


---> Running TabICL on: Diabetes (Endocrinology) <---


Diabetes Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Diabetes recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Diabetes,560.0,140.0,8,Endocrinology,0.874554,0.135889,0.10563,0.074779,0.070085,...,0.008857,0.007294,0.02,0.025455,0.01189,0.021787,0.082857,0.016444,0.069437,0.006638


In [9]:
run_single_dataset(
    name="Stroke",
    path="/content/healthcare-dataset-stroke-data_preprocessed.csv",
    domain="Neurology",
)


---> Running TabICL on: Stroke (Neurology) <---


Stroke Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Stroke recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Stroke,560.0,140.0,10,Neurology,0.901108,0.080787,0.123608,0.089743,0.037567,...,0.004541,0.003286,0.015714,0.020454,-0.016437,0.027548,0.084286,0.019166,0.084183,0.015015


In [10]:
run_single_dataset(
    name="Heart Disease",
    path="/content/heart_disease_uci_preprocessed.csv",
    domain="Cardiology",
)


---> Running TabICL on: Heart Disease (Cardiology) <---


Heart Disease Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Heart Disease recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Heart Disease,560.0,140.0,14,Cardiology,0.271163,0.052593,0.233218,0.126956,0.4522,...,0.00302,0.005319,0.007143,0.029881,0.005794,0.024339,0.044286,0.015485,0.110044,0.010133


In [11]:
run_single_dataset(
    name="Hepatitis",
    path="/content/hepatitis_data_preprocessed.csv",
    domain="Infectious Disease",
)


---> Running TabICL on: Hepatitis (Infectious Disease) <---


Hepatitis Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Hepatitis recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Hepatitis,124.0,31.0,19,Infectious Disease,0.115385,0.258008,NaN,NaN,0.182812,...,0.003714,0.02462,-0.006452,0.014426,-0.02,0.044721,0.019355,0.028852,0.135911,0.012712


In [12]:
run_single_dataset(
    name="Liver Disease",
    path="/content/liver_patient_dataset_preprocessed.csv",
    domain="Hepatology",
)


---> Running TabICL on: Liver Disease (Hepatology) <---


Liver Disease Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Liver Disease recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Liver Disease,560.0,140.0,10,Hepatology,0.869251,0.064008,0.069445,0.048438,0.073046,...,0.216816,0.029661,0.308571,0.018489,0.207779,0.020478,0.397143,0.014812,0.30555,0.010674


In [13]:
run_single_dataset(
    name="Thyroid",
    path="/content/thyroid_dataset_preprocessed.csv",
    domain="Endocrinology",
)


---> Running TabICL on: Thyroid (Endocrinology) <---


Thyroid Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Thyroid recorded in TabICL_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Thyroid,560.0,140.0,21,Endocrinology,0.76284,0.11904,0.003509,0.007846,0.131691,...,0.150327,0.047743,0.198571,0.040216,0.209391,0.044466,0.212857,0.031703,0.299854,0.034669


In [14]:
# Code to sort all 10 datasets into the standard order and download
import os
import pandas as pd
from google.colab import files

CSV_PATH = "TabICL_Results.csv"

if not os.path.exists(CSV_PATH):
    print(f"Error: {CSV_PATH} not found in /content/.")
else:
    df = pd.read_csv(CSV_PATH)

    # Standard order matching Cell 1 / Project Context
    standard_order = [
        "COPD",
        "Alzheimers",
        "Arrhythmia",
        "Autism",
        "Diabetes",
        "Stroke",
        "Heart Disease",
        "Hepatitis",
        "Liver Disease",
        "Thyroid",
    ]

    # Re-order rows to match the standard order
    df["Dataset"] = pd.Categorical(
        df["Dataset"], categories=standard_order, ordered=True
    )
    df = df.sort_values("Dataset").reset_index(drop=True)

    # Save cleanly with 6 decimal places as required by protocol
    df.to_csv(CSV_PATH, index=False, float_format="%.6f")

    print(f"Successfully combined and sorted {len(df)}/10 datasets!")
    display(df[["Dataset", "Domain", "Train_N", "Test_N", "Num_Feats"]])

    # Trigger automatic download
    files.download(CSV_PATH)

Successfully combined and sorted 10/10 datasets!


,Dataset,Domain,Train_N,Test_N,Num_Feats
0,COPD,Pulmonology,504.0,126.0,14
1,Alzheimers,Neurology,560.0,140.0,32
2,Arrhythmia,Cardiology,361.6,90.4,262
3,Autism,Psychiatry,559.2,139.8,19
4,Diabetes,Endocrinology,560.0,140.0,8
5,Stroke,Neurology,560.0,140.0,10
6,Heart Disease,Cardiology,560.0,140.0,14
7,Hepatitis,Infectious Disease,124.0,31.0,19
8,Liver Disease,Hepatology,560.0,140.0,10
9,Thyroid,Endocrinology,560.0,140.0,21


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>